# تحلیل ادعای هزینه

این دفترچه نشان می‌دهد چگونه می‌توان عامل‌هایی ایجاد کرد که با استفاده از افزونه‌ها هزینه‌های سفر را از تصاویر رسید محلی پردازش کنند، یک ایمیل ادعای هزینه تولید کنند و داده‌های هزینه را با استفاده از نمودار دایره‌ای مصورسازی کنند. عامل‌ها بر اساس زمینه کار به صورت پویا توابع را انتخاب می‌کنند.

مراحل:
1. عامل OCR تصویر رسید محلی را پردازش کرده و داده‌های هزینه سفر را استخراج می‌کند.
2. عامل ایمیل یک ایمیل ادعای هزینه تولید می‌کند.

### نمونه‌ای از یک سناریوی هزینه سفر:
تصور کنید شما یک کارمند هستید که برای یک جلسه کاری به شهر دیگری سفر می‌کنید. شرکت شما سیاستی دارد که تمام هزینه‌های معقول مرتبط با سفر را بازپرداخت می‌کند. اینجا تفکیک هزینه‌های احتمالی سفر آمده است:
- حمل و نقل:
بلیط هواپیما برای سفر رفت و برگشت از شهر محل سکونت به شهر مقصد.
تاکسی یا خدمات حمل و نقل اینترنتی به فرودگاه و از فرودگاه.
حمل و نقل محلی در شهر مقصد (مانند حمل و نقل عمومی، خودروهای اجاره‌ای یا تاکسی‌ها).

- اقامت:
اقامت در هتلی با قیمت متوسط برای سه شب نزدیک محل جلسه.

- غذا:
سهمیه روزانه غذا برای صبحانه، ناهار و شام بر اساس سیاست روزانه شرکت.

- هزینه‌های متفرقه:
هزینه پارکینگ در فرودگاه.
هزینه دسترسی به اینترنت در هتل.
انعام یا هزینه‌های خدمات کوچک.

- مستندسازی:
شما تمام رسیدها (پروازها، تاکسی‌ها، هتل، غذا و غیره) و گزارش هزینه کامل شده را برای بازپرداخت ارائه می‌دهید.


## وارد کردن کتابخانه‌های مورد نیاز

کتابخانه‌ها و ماژول‌های لازم برای دفترچه یادداشت را وارد کنید.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## تعریف مدل‌های هزینه

 یک مدل پایدانتیک برای هزینه‌های فردی ایجاد کنید و یک کلاس ExpenseFormatter برای تبدیل پرسش کاربر به داده‌های ساختاری هزینه‌ها بسازید.

 هر هزینه به فرمت زیر نمایش داده خواهد شد:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## تعریف ابزارها - تولید ایمیل

یک تابع ابزار برای تولید یک ایمیل جهت ارسال ادعای هزینه ایجاد کنید.
- این ابزار از دکوراتور `@tool` از چارچوب عامل مایکروسافت استفاده می‌کند.
- این ابزار مجموع مبلغ هزینه‌ها را محاسبه و جزئیات را به صورت بدنه ایمیل قالب‌بندی می‌کند.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# بارگزاری تصاویر رسید هزینه سفر

یک تابع کمکی برای بارگزاری تصاویر رسید به عنوان محتوای چندرسانه‌ای بومی ایجاد کنید.
- این تابع بایت‌های تصویر رسید را می‌خواند و آن‌ها را در قالب چارچوب عامل `Content` برای تحلیل توسط عامل بسته‌بندی می‌کند.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## پردازش هزینه‌ها

عوامل را تعریف کرده و آن‌ها را با استفاده از `WorkflowBuilder` در یک جریان کاری ترتیبی متصل کنید.
- عامل OCR داده‌های ساختاریافته هزینه را از تصویر رسید موجود در پیام کاربر استخراج می‌کند.
- عامل ایمیل داده‌های استخراج شده را دریافت کرده و یک ایمیل ادعای هزینه حرفه‌ای با استفاده از ابزار `generate_expense_email` تولید می‌کند.
- `WorkflowBuilder` با `add_edge` یک خط لوله ترتیبی ایجاد می‌کند: عامل OCR → عامل ایمیل.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## تابع اصلی

جریان کاری ترتیبی را بسازید و آن را اجرا کنید تا تصویر رسید پردازش شود و ایمیل درخواست هزینه تولید شود.


> **توجه:** بایت‌های رسید در چارچوب چندرسانه‌ای Agent Framework بومی `Content` بسته‌بندی شده‌اند، بنابراین Foundry آن‌ها را به‌عنوان ورودی تصویر دریافت می‌کند نه متن عادی.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
